# EFIplus Mediterranean Dataset — PCA, PCoA and LDA

This notebook solves the exercise using the dataset `EFIplus_medit.zip` / `EFIplus_medit (2).zip`.

The analysis uses only sites from the **Douro**, **Tejo**, **Mondego** and **Minho** basins, and uses `Catchment_name` as the grouping variable in all plots.

The quantitative environmental variables used are the same as in the previous exercise:

- `Altitude`
- `Actual_river_slope`
- `Elevation_mean_catch`
- `prec_ann_catch`
- `temp_ann`
- `temp_jan`
- `temp_jul`


In [ ]:
# Import libraries
# These libraries are enough for PCA, PCoA, LDA, data manipulation and plotting.

import os
import zipfile
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from scipy.spatial.distance import pdist, squareform

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 120)


## 1. Load the dataset

Upload `EFIplus_medit.zip` or `EFIplus_medit (2).zip` to Colab, or place it in the same folder as this notebook. The code below searches automatically for a ZIP file containing `EFIplus_medit` in its name.


In [ ]:
# In Google Colab, uncomment and run this cell if the ZIP file is not already uploaded.
# from google.colab import files
# uploaded = files.upload()


In [ ]:
# Find the EFIplus zip file automatically.
# This avoids errors if the file is called 'EFIplus_medit.zip' or 'EFIplus_medit (2).zip'.

possible_zips = list(Path('.').glob('**/EFIplus_medit*.zip'))

if len(possible_zips) == 0:
    raise FileNotFoundError(
        "No EFIplus_medit ZIP file was found. Upload 'EFIplus_medit.zip' or 'EFIplus_medit (2).zip' and run again."
    )

zip_path = possible_zips[0]
print(f"Using ZIP file: {zip_path}")

extract_dir = Path('EFIplus_medit_extracted')
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

# Show files extracted from the ZIP.
extracted_files = list(extract_dir.glob('**/*'))
[f.name for f in extracted_files if f.is_file()][:20]


In [ ]:
# Read the dataset.
# The code searches for CSV/TXT files inside the extracted ZIP and reads the largest one,
# which is normally the main EFIplus dataset.

candidate_files = [f for f in extract_dir.glob('**/*') if f.suffix.lower() in ['.csv', '.txt', '.tsv']]

if len(candidate_files) == 0:
    raise FileNotFoundError("No CSV/TXT file was found inside the ZIP.")

# Choose the largest file, because it is usually the full data table.
data_file = max(candidate_files, key=lambda x: x.stat().st_size)
print(f"Reading data file: {data_file}")

# sep=None lets pandas infer whether the separator is comma, semicolon or tab.
df = pd.read_csv(data_file, sep=None, engine='python')

print(df.shape)
df.head()


## 2. Select basins and quantitative environmental variables

The exercise asks for only the Douro, Tejo, Mondego and Minho basins. Missing values in the selected variables are removed because PCA, PCoA and LDA require complete numerical data.


In [ ]:
# Basins required in the exercise
selected_basins = ['Douro', 'Tejo', 'Mondego', 'Minho']

# Quantitative environmental variables from the previous exercise
env_vars = [
    'Altitude',
    'Actual_river_slope',
    'Elevation_mean_catch',
    'prec_ann_catch',
    'temp_ann',
    'temp_jan',
    'temp_jul'
]

# Check that all required columns exist before continuing.
required_cols = ['Catchment_name'] + env_vars
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"These required columns are missing from the dataset: {missing_cols}")

# Filter the dataset to the required basins only.
data = df[df['Catchment_name'].isin(selected_basins)].copy()

# Convert environmental variables to numeric, forcing problematic values to NaN.
for col in env_vars:
    data[col] = pd.to_numeric(data[col], errors='coerce')

# Remove rows with missing values in the variables used for the analyses.
data_clean = data.dropna(subset=required_cols).copy()

print("Original filtered data shape:", data.shape)
print("Clean data shape:", data_clean.shape)
print("Number of sites by catchment:")
display(data_clean['Catchment_name'].value_counts())

data_clean[['Site_code', 'Catchment_name'] + env_vars].head()


In [ ]:
# Standardise the quantitative environmental variables.
# Standardisation is important because variables are measured in different units.

X = data_clean[env_vars].values
y = data_clean['Catchment_name'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=env_vars, index=data_clean.index)
X_scaled_df.head()


# Part 1 — PCA based on quantitative environmental variables

PCA reduces the environmental variables into synthetic axes that summarise the main environmental gradients among sites. The biplot shows both the position of sites and the contribution of each environmental variable.


In [ ]:
# Run PCA with two components.

pca = PCA(n_components=2)
pca_scores = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(pca_scores, columns=['PC1', 'PC2'], index=data_clean.index)
pca_df['Catchment_name'] = y

# PCA loadings show how strongly each original variable contributes to the PCA axes.
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=env_vars
)

explained = pca.explained_variance_ratio_ * 100
print(f"PC1 explains {explained[0]:.2f}% of the variance")
print(f"PC2 explains {explained[1]:.2f}% of the variance")

display(loadings)


In [ ]:
# PCA biplot grouped by Catchment_name.

fig, ax = plt.subplots(figsize=(10, 8))

# Plot sites by catchment.
for catchment in selected_basins:
    subset = pca_df[pca_df['Catchment_name'] == catchment]
    ax.scatter(subset['PC1'], subset['PC2'], label=catchment, alpha=0.75)

# Scale arrows so that they fit nicely in the same figure as the scores.
arrow_scale = min(
    (pca_df['PC1'].max() - pca_df['PC1'].min()),
    (pca_df['PC2'].max() - pca_df['PC2'].min())
) * 0.35

for var in env_vars:
    ax.arrow(
        0, 0,
        loadings.loc[var, 'PC1'] * arrow_scale,
        loadings.loc[var, 'PC2'] * arrow_scale,
        head_width=0.06,
        length_includes_head=True
    )
    ax.text(
        loadings.loc[var, 'PC1'] * arrow_scale * 1.12,
        loadings.loc[var, 'PC2'] * arrow_scale * 1.12,
        var,
        fontsize=10
    )

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)
ax.set_xlabel(f"PC1 ({explained[0]:.1f}% variance explained)")
ax.set_ylabel(f"PC2 ({explained[1]:.1f}% variance explained)")
ax.set_title("PCA biplot of environmental variables by catchment")
ax.legend(title='Catchment_name')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pca_biplot_catchment.png', dpi=300, bbox_inches='tight')
plt.show()


# Part 2 — PCoA based on the same environmental variables

PCoA is based on a distance matrix between sites. Here, Euclidean distance is calculated using the same standardised quantitative environmental variables. The first two PCoA axes are then plotted and grouped by `Catchment_name`.


In [ ]:
# Function to run classical PCoA from a distance matrix.
# This implementation follows the standard double-centering procedure.

def pcoa(distance_matrix):
    n = distance_matrix.shape[0]
    D2 = distance_matrix ** 2
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D2 @ J

    eigvals, eigvecs = np.linalg.eigh(B)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    # Keep only positive eigenvalues for real coordinates.
    positive = eigvals > 0
    eigvals_pos = eigvals[positive]
    eigvecs_pos = eigvecs[:, positive]

    coords = eigvecs_pos * np.sqrt(eigvals_pos)
    variance_explained = eigvals_pos / eigvals_pos.sum() * 100

    return coords, eigvals_pos, variance_explained

# Euclidean distance matrix between sites.
dist_matrix = squareform(pdist(X_scaled, metric='euclidean'))

pcoa_coords, pcoa_eigvals, pcoa_var = pcoa(dist_matrix)

pcoa_df = pd.DataFrame(
    pcoa_coords[:, :2],
    columns=['PCoA1', 'PCoA2'],
    index=data_clean.index
)
pcoa_df['Catchment_name'] = y

print(f"PCoA1 explains {pcoa_var[0]:.2f}% of the variation in distances")
print(f"PCoA2 explains {pcoa_var[1]:.2f}% of the variation in distances")
pcoa_df.head()


In [ ]:
# PCoA plot grouped by Catchment_name.

fig, ax = plt.subplots(figsize=(10, 8))

for catchment in selected_basins:
    subset = pcoa_df[pcoa_df['Catchment_name'] == catchment]
    ax.scatter(subset['PCoA1'], subset['PCoA2'], label=catchment, alpha=0.75)

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)
ax.set_xlabel(f"PCoA1 ({pcoa_var[0]:.1f}% variation explained)")
ax.set_ylabel(f"PCoA2 ({pcoa_var[1]:.1f}% variation explained)")
ax.set_title("PCoA of environmental variables by catchment")
ax.legend(title='Catchment_name')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pcoa_catchment.png', dpi=300, bbox_inches='tight')
plt.show()


# Part 3 — Linear Discriminant Analysis based on quantitative environmental variables

LDA is supervised: the catchment basins are used as pre-defined groups. The analysis finds linear combinations of environmental variables that maximise separation among Douro, Tejo, Mondego and Minho sites.


In [ ]:
# Run LDA using Catchment_name as the pre-defined group.
# With four groups, the maximum number of discriminant axes is three.
# For the biplot, we use the first two axes.

lda = LinearDiscriminantAnalysis(n_components=2)
lda_scores = lda.fit_transform(X_scaled, y)

lda_df = pd.DataFrame(lda_scores, columns=['LD1', 'LD2'], index=data_clean.index)
lda_df['Catchment_name'] = y

lda_explained = lda.explained_variance_ratio_ * 100
print(f"LD1 explains {lda_explained[0]:.2f}% of the discriminant variance")
print(f"LD2 explains {lda_explained[1]:.2f}% of the discriminant variance")

# Coefficients indicate the contribution of each environmental variable to the discriminant axes.
lda_loadings = pd.DataFrame(
    lda.scalings_[:, :2],
    columns=['LD1', 'LD2'],
    index=env_vars
)

display(lda_loadings)


In [ ]:
# LDA biplot grouped by Catchment_name.

fig, ax = plt.subplots(figsize=(10, 8))

for catchment in selected_basins:
    subset = lda_df[lda_df['Catchment_name'] == catchment]
    ax.scatter(subset['LD1'], subset['LD2'], label=catchment, alpha=0.75)

# Scale variable arrows to fit the discriminant score space.
arrow_scale_lda = min(
    (lda_df['LD1'].max() - lda_df['LD1'].min()),
    (lda_df['LD2'].max() - lda_df['LD2'].min())
) * 0.15

for var in env_vars:
    ax.arrow(
        0, 0,
        lda_loadings.loc[var, 'LD1'] * arrow_scale_lda,
        lda_loadings.loc[var, 'LD2'] * arrow_scale_lda,
        head_width=0.06,
        length_includes_head=True
    )
    ax.text(
        lda_loadings.loc[var, 'LD1'] * arrow_scale_lda * 1.12,
        lda_loadings.loc[var, 'LD2'] * arrow_scale_lda * 1.12,
        var,
        fontsize=10
    )

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)
ax.set_xlabel(f"LD1 ({lda_explained[0]:.1f}% discriminant variance)")
ax.set_ylabel(f"LD2 ({lda_explained[1]:.1f}% discriminant variance)")
ax.set_title("LDA biplot of environmental variables by catchment")
ax.legend(title='Catchment_name')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lda_biplot_catchment.png', dpi=300, bbox_inches='tight')
plt.show()


## Short interpretation template

After running the notebook, use the values printed above to complete the interpretation. A possible structure is:

> The PCA biplot shows the main environmental gradients among the selected basins. Sites positioned close to one another have similar environmental conditions, while sites farther apart are more different. The arrows indicate which variables contribute most strongly to the PCA axes. Longer arrows represent variables with stronger contributions.
>
> The PCoA plot projects sites based on their Euclidean environmental distances. The grouping by `Catchment_name` allows visual assessment of whether sites from the same basin cluster together or overlap with other basins.
>
> The LDA biplot uses the basin identity as a pre-defined grouping variable. Therefore, it is expected to show clearer separation between catchments than PCA or PCoA, because LDA maximises among-group differences. Variables with larger arrows are the most important for discriminating Douro, Tejo, Mondego and Minho.
